In [ ]:
import sys
from pathlib import Path

_src = Path("../src").resolve()
if str(_src) not in sys.path:
    sys.path.insert(0, str(_src))


# Cont-Kokholm (2014) — Model Replication

Replicates the key results of **Cont & Kokholm, "A consistent pricing model for index options
and volatility derivatives" (2014)** using the `finpricing` library.

**Contents**
1. Paper parameters (Table 2)
2. Model validation — martingale properties
3. VIX option implied-volatility smile (6 maturities)
4. VIX implied-volatility term structure
5. Index option implied-volatility surface (6 tenors)
6. Merton vs Kou comparison
7. Jump size distributions


In [ ]:
import warnings
import time
import numpy as np
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
plt.style.use("ggplot")
plt.rcParams.update({"figure.dpi": 110, "font.size": 10})

from finpricing.models.vix.model import VixModel, get_UTm, get_sigmas
from finpricing.data.fixtures import (
    TENOR_DATES, VI_0,
    MERTON_PARAMS, MERTON_B_I,
    KOU_PARAMS, KOU_B_I,
)
from finpricing.models.black_scholes.implied_volatility import (
    ImpliedVolatilityCalculator, ImpliedVolatilityParams, Method
)

MATURITY_LABELS = [f"{int(round(T * 12))}M" for T in TENOR_DATES[1:]]
R = 0.0  # risk-free rate (paper uses 0)
print("finpricing loaded OK")


## 1 · Model Parameters (Table 2)

In [ ]:
print("Tenor dates (months):", (TENOR_DATES * 12).astype(int))
print("V^i_0 (initial variance swap rates):", VI_0)
print()
print("Merton  |", "  ".join(f"{k}={v}" for k, v in MERTON_PARAMS.model_dump().items()))
print("  b_i  =", MERTON_B_I)
print()
print("Kou     |", "  ".join(f"{k}={v}" for k, v in KOU_PARAMS.model_dump().items()))
print("  b_i  =", KOU_B_I)


## 2 · Model Validation — Martingale Properties

Two checks from the paper:
- **E[V^i_{T_i} / V^i_0] ≈ 1** : each forward variance swap rate is a Q-martingale
- **E[U_{T_k}] ≈ 1** : the index price ratio S_{T_k}/S_0 has expectation 1 under Q

Set `NUM_PATHS = 2_000_000` to reproduce the paper's accuracy (~1 min on CPU).


In [ ]:
NUM_PATHS = 200_000  # paper uses 2_000_000

merton = VixModel(model_type="Merton")
kou    = VixModel(model_type="Kou")

t0 = time.time()
merton.store_tenor_data(NUM_PATHS)
kou.store_tenor_data(NUM_PATHS)
print(f"Simulation: {time.time()-t0:.1f}s  ({NUM_PATHS:,} paths each)")

def martingale_check(model, params, b_i, label):
    _, diff_Z, jumps_S, _, VTi = model._stored_data
    k = len(TENOR_DATES) - 1
    sigmas_sq = get_sigmas(VI_0, VTi, model.constraint_cf, b_i)
    U = get_UTm(VI_0[:k], VTi[:, :k], diff_Z[:, :k], jumps_S[:, :k],
                sigmas_sq[:, :k], b_i[:k], TENOR_DATES[:k+1], params)
    print(f"\n{label}")
    print(f"  E[V^i_Ti / V^i_0] = {np.round(np.mean(VTi/VI_0, axis=0), 4)}")
    print(f"  E[U_T{k}]         = {np.mean(U):.5f}  (should be ≈ 1)")

martingale_check(merton, MERTON_PARAMS, MERTON_B_I, "Merton")
martingale_check(kou,    KOU_PARAMS,    KOU_B_I,    "Kou")


In [ ]:
_, _, _, _, VTi_m = merton._stored_data
_, _, _, _, VTi_k = kou._stored_data

fig, ax = plt.subplots(figsize=(7, 3.5))
ax.plot(TENOR_DATES[1:]*12, VI_0,
        "ko-", ms=8, zorder=3, label=r"$V^i_0$  (paper initial values)")
ax.plot(TENOR_DATES[1:]*12, np.mean(VTi_m, axis=0),
        "bs--", ms=5, label=r"$E[V^i_{T_i}]$  Merton")
ax.plot(TENOR_DATES[1:]*12, np.mean(VTi_k, axis=0),
        "r^--", ms=5, label=r"$E[V^i_{T_i}]$  Kou")
ax.set_xlabel("Maturity (months)")
ax.set_ylabel("Forward variance swap rate")
ax.set_title("Variance Swap Term Structure")
ax.legend()
plt.tight_layout()
plt.show()


## 3 · VIX Option Implied-Volatility Smile

In [ ]:
def bs_iv(price, S, K, T, r=0.0):
    if price <= 1e-8 or not np.isfinite(price):
        return np.nan
    try:
        p = ImpliedVolatilityParams(call_value=float(price), s=S, k=K, t=T, r=r,
                                    method=Method.BRENTQ)
        iv = ImpliedVolatilityCalculator.implied_volatility(p)
        return iv if 0.01 < iv < 5.0 else np.nan
    except Exception:
        return np.nan


In [ ]:
V0 = 1.0  # normalised VIX level; x-axis = K/V0

fig, axs = plt.subplots(3, 2, figsize=(11, 12))
fig.suptitle("VIX Option Implied Volatility Smile\n"
             "(Cont-Kokholm calibrated parameters, Table 2)", fontsize=13)

for j, ax in enumerate(axs.flat):
    T  = TENOR_DATES[j + 1]
    ks = np.arange(0.8 if j == 0 else 0.7, 2.6, 0.02)

    for model, label, color in [
        (merton, "Merton", "steelblue"),
        (kou,    "Kou",    "darkorange"),
    ]:
        prices = [model.vix_option_pricer(V0, K, T, r=R) for K in ks]
        ivs    = np.array([bs_iv(p, V0, K, T, R) for p, K in zip(prices, ks)])
        mask   = ~np.isnan(ivs)
        ax.plot(ks[mask], ivs[mask], lw=1.8, label=label, color=color)

    ax.set_title(f"T = {MATURITY_LABELS[j]}")
    ax.set_xlabel("K / V$_0$  (moneyness)")
    ax.set_ylabel("Implied vol")
    ax.legend(fontsize=8)

plt.tight_layout()
plt.show()


## 4 · VIX Implied-Volatility Term Structure (ATM)

In [ ]:
V0 = 1.0
atm_ivs_m, atm_ivs_k, atm_p_m, atm_p_k = [], [], [], []

for T in TENOR_DATES[1:]:
    pm = merton.vix_option_pricer(V0, K=V0, T=T, r=R)
    pk = kou.vix_option_pricer(V0, K=V0, T=T, r=R)
    atm_p_m.append(pm);  atm_p_k.append(pk)
    atm_ivs_m.append(bs_iv(pm, V0, V0, T, R))
    atm_ivs_k.append(bs_iv(pk, V0, V0, T, R))

mats = TENOR_DATES[1:] * 12

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))

ax1.plot(mats, atm_p_m, "o-", color="steelblue",  label="Merton")
ax1.plot(mats, atm_p_k, "s-", color="darkorange", label="Kou")
ax1.set_title("ATM VIX Call Price");  ax1.set_xlabel("Maturity (months)")
ax1.set_ylabel("Price  (V$_0$ = 1)"); ax1.legend()

ax2.plot(mats, atm_ivs_m, "o-", color="steelblue",  label="Merton")
ax2.plot(mats, atm_ivs_k, "s-", color="darkorange", label="Kou")
ax2.set_title("ATM VIX Implied Volatility"); ax2.set_xlabel("Maturity (months)")
ax2.set_ylabel("Implied vol"); ax2.legend()

plt.tight_layout(); plt.show()


## 5 · Index Option Implied-Volatility Surface

Uses the MC paths already stored in Section 2.  
Increase `NUM_PATHS` to 2 × 10⁶ (re-run from top) for paper-quality smoothness.


In [ ]:
strikes_per_tenor = [
    np.arange(0.70, 1.15, 0.02),
    np.arange(0.65, 1.17, 0.02),
    np.arange(0.78, 1.18, 0.02),
    np.arange(0.50, 1.35, 0.02),
    np.arange(0.64, 1.35, 0.02),
    np.arange(0.55, 1.44, 0.02),
]

fig, axs = plt.subplots(3, 2, figsize=(11, 12))
fig.suptitle("Index Option Implied Volatility — Merton Model\n"
             "(Cont-Kokholm calibrated parameters, Table 2)", fontsize=13)

for j, ax in enumerate(axs.flat):
    T  = TENOR_DATES[j + 1]
    ks = strikes_per_tenor[j]

    prices = merton.index_option_pricer(S0=1.0, strikes=ks, tenor_index=j+1, r=R)
    ivs    = np.array([bs_iv(prices[i], 1.0, ks[i], T, R) for i in range(len(ks))])
    mask   = ~np.isnan(ivs) & (ivs > 0.01)

    ax.plot(ks[mask], ivs[mask], lw=1.8, color="steelblue")
    ax.set_title(f"T = {MATURITY_LABELS[j]}")
    ax.set_xlabel("K / S$_0$"); ax.set_ylabel("Implied vol")
    ax.set_yticks(np.arange(0.1, 0.6, 0.1))

plt.tight_layout(); plt.show()


## 6 · Merton vs Kou — Index Option Smile

In [ ]:
fig, axs = plt.subplots(3, 2, figsize=(11, 12))
fig.suptitle("Index Option Implied Volatility — Merton vs Kou\n"
             "(Cont-Kokholm calibrated parameters, Table 2)", fontsize=13)

for j, ax in enumerate(axs.flat):
    T  = TENOR_DATES[j + 1]
    ks = strikes_per_tenor[j]

    for model, label, color in [
        (merton, "Merton", "steelblue"),
        (kou,    "Kou",    "darkorange"),
    ]:
        prices = model.index_option_pricer(S0=1.0, strikes=ks, tenor_index=j+1, r=R)
        ivs    = np.array([bs_iv(prices[i], 1.0, ks[i], T, R) for i in range(len(ks))])
        mask   = ~np.isnan(ivs) & (ivs > 0.01)
        ax.plot(ks[mask], ivs[mask], lw=1.8, label=label, color=color)

    ax.set_title(f"T = {MATURITY_LABELS[j]}")
    ax.set_xlabel("K / S$_0$"); ax.set_ylabel("Implied vol")
    ax.set_yticks(np.arange(0.1, 0.6, 0.1)); ax.legend(fontsize=8)

plt.tight_layout(); plt.show()


## 7 · Jump Size Distributions

In [ ]:
_, _, _, jumps_V_m, _ = merton._stored_data
_, _, _, jumps_V_k, _ = kou._stored_data

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
fig.suptitle("Cumulative VIX jump term in first tenor period", fontsize=12)

for ax, data, label, color in [
    (ax1, jumps_V_m[:, 0], "Merton (Gaussian jumps)",        "steelblue"),
    (ax2, jumps_V_k[:, 0], "Kou (double-exponential jumps)", "darkorange"),
]:
    ax.hist(data, bins=80, density=True, color=color, alpha=0.8, edgecolor="none")
    ax.set_title(label); ax.set_xlabel("Jump term"); ax.set_ylabel("Density")
    ax.axvline(0, color="k", lw=0.8, ls="--")

plt.tight_layout(); plt.show()

print(f"Merton:  mean={jumps_V_m[:,0].mean():.4f}  std={jumps_V_m[:,0].std():.4f}")
print(f"Kou:     mean={jumps_V_k[:,0].mean():.4f}  std={jumps_V_k[:,0].std():.4f}")
